<a href="https://colab.research.google.com/github/tu702019/Machine_Learning_Algorithm_and_Its_Application/blob/main/HW1_diamonds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Libraries & Data Information

In [161]:
# Import Libraries
import pandas as pd
from sklearn.feature_selection import SelectPercentile, f_regression

In [162]:
# load the data
url_diamonds = 'https://raw.githubusercontent.com/tu702019/Machine_Learning_Algorithm_and_Its_Application/refs/heads/main/HW1_Data%20Preprocessing/daimonds(predict%20price).csv'
df_diamonds = pd.read_csv(url_diamonds)

## Data Information

In [163]:
#Print the first 5 rows of the data
df_diamonds.head()

,Unnamed: 0.1,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,0,1,0.23,NaN,E,SI2,61.5,55.0,326.0,3.95,3.98,2.43
1,1,2,0.21,Premium,E,SI1,59.8,NaN,326.0,3.89,3.84,2.31
2,2,3,0.23,Good,E,VS1,56.9,65.0,327.0,4.05,4.07,2.31
3,3,4,0.29,Premium,I,NaN,62.4,58.0,334.0,4.20,NaN,2.63
4,4,5,0.31,Good,J,SI2,63.3,58.0,335.0,4.34,4.35,2.75


In [164]:
# count the number of rows and coloumns in the dataset
df_diamonds.shape

(53940, 12)

In [165]:
df_diamonds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Unnamed: 0.1  53940 non-null  int64  
 1   Unnamed: 0    53940 non-null  int64  
 2   carat         52947 non-null  float64
 3   cut           52951 non-null  object 
 4   color         52948 non-null  object 
 5   clarity       52945 non-null  object 
 6   depth         52950 non-null  float64
 7   table         52947 non-null  float64
 8   price         52948 non-null  float64
 9   x             52949 non-null  float64
 10  y             52943 non-null  float64
 11  z             52948 non-null  float64
dtypes: float64(7), int64(2), object(3)
memory usage: 4.9+ MB


# Data Preprocessing

## Count the missing values

In [166]:
# Count the number of missing values in each column
missing_counts = df_diamonds.isnull().sum()

# Filter out columns that contain missing values
features_with_missing = missing_counts[missing_counts > 0]

print(f"Missing values are distributed across {len(features_with_missing)} features.")
print("Numbers of missing value:",df_diamonds.isnull().sum().sum())
print("Features with missing values:\n", features_with_missing)

Missing values are distributed across 10 features.
Numbers of missing value: 9924
Features with missing values:
 carat      993
cut        989
color      992
clarity    995
depth      990
table      993
price      992
x          991
y          997
z          992
dtype: int64


## Null/Missing Value Estimation

In [167]:
# Fill missing values based on data type
df_diamonds_filled = df_diamonds.copy()

for column in features_with_missing.index:
    # Categorical variables: Fill with mode
    if df_diamonds[column].dtype == 'object':
        df_diamonds_filled[column] = df_diamonds[column].fillna(df_diamonds[column].mode()[0])
    # Numerical variables: Fill with mean
    else:
        df_diamonds_filled[column] = df_diamonds[column].fillna(df_diamonds[column].mean())

In [168]:
# Drop unnecessary columns
df_diamonds_filled = df_diamonds_filled.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'], errors='ignore')

In [169]:
# check
print("numbers of missing value after filling:", df_diamonds_filled.isnull().sum().sum())

numbers of missing value after filling: 0


# Categorical-Numerical Feature Transformation

In [170]:
# Split features and target
X = df_diamonds_filled.drop('price', axis=1)
y = df_diamonds_filled['price']

In [171]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 9 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   x        53940 non-null  float64
 7   y        53940 non-null  float64
 8   z        53940 non-null  float64
dtypes: float64(6), object(3)
memory usage: 3.7+ MB


In [172]:
original_features_diamonds = X.shape[1]
print(f"Number of original features:", original_features_diamonds)

Number of original features: 9


In [173]:
# One-Hot Encoding for categorical variables
X_encoded = pd.get_dummies(X, drop_first=True)

In [174]:
encoded_features_diamonds = X_encoded.shape[1]
print(f"Number of encoded features:", encoded_features_diamonds)

Number of encoded features: 23


In [175]:
X_encoded.head()

,carat,depth,table,x,y,z,cut_Good,cut_Ideal,cut_Premium,cut_Very Good,...,color_H,color_I,color_J,clarity_IF,clarity_SI1,clarity_SI2,clarity_VS1,clarity_VS2,clarity_VVS1,clarity_VVS2
0,0.23,61.5,55.000000,3.95,3.980000,2.43,False,True,False,False,...,False,False,False,False,False,True,False,False,False,False
1,0.21,59.8,57.457393,3.89,3.840000,2.31,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
2,0.23,56.9,65.000000,4.05,4.070000,2.31,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
3,0.29,62.4,58.000000,4.20,5.734215,2.63,False,False,True,False,...,False,True,False,False,True,False,False,False,False,False
4,0.31,63.3,58.000000,4.34,4.350000,2.75,True,False,False,False,...,False,False,True,False,False,True,False,False,False,False


In [176]:
X_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   carat          53940 non-null  float64
 1   depth          53940 non-null  float64
 2   table          53940 non-null  float64
 3   x              53940 non-null  float64
 4   y              53940 non-null  float64
 5   z              53940 non-null  float64
 6   cut_Good       53940 non-null  bool   
 7   cut_Ideal      53940 non-null  bool   
 8   cut_Premium    53940 non-null  bool   
 9   cut_Very Good  53940 non-null  bool   
 10  color_E        53940 non-null  bool   
 11  color_F        53940 non-null  bool   
 12  color_G        53940 non-null  bool   
 13  color_H        53940 non-null  bool   
 14  color_I        53940 non-null  bool   
 15  color_J        53940 non-null  bool   
 16  clarity_IF     53940 non-null  bool   
 17  clarity_SI1    53940 non-null  bool   
 18  clarit

# Feature Selection

In [177]:
# Select top 60% features based on ANOVA F-value
selector = SelectPercentile(score_func=f_regression, percentile=60)
X_selected = selector.fit_transform(X_encoded, y)

# Get selected feature names
selected_features = X_encoded.columns[selector.get_support()]
print("Selected features (top 60% based on F-regression scores):")
print(selected_features.tolist())
print(f"Number of selected features: {len(selected_features)}")

Selected features (top 60% based on F-regression scores):
['carat', 'table', 'x', 'y', 'z', 'cut_Ideal', 'cut_Premium', 'color_E', 'color_H', 'color_I', 'color_J', 'clarity_SI2', 'clarity_VVS1', 'clarity_VVS2']
Number of selected features: 14


# Save to Excel (.xlsx) & CSV (.csv)

In [178]:
# Step 1: Convert X_selected to DataFrame with column names
X_selected_df = pd.DataFrame(X_selected, columns=selected_features)

# Step 2: Combine with target variable
final_df = pd.concat([X_selected_df, pd.Series(y, name='price')], axis=1)

# Save to Excel (.xlsx)
final_df.to_excel("HW1_diamonds.xlsx", index=False)

# Save to CSV (.csv)
final_df.to_csv("HW1_diamondss.csv", index=False)

print("Saved to: HW1_diamonds.xlsx and HW1_diamonds.csv")


Saved to: HW1_diamonds.xlsx and HW1_diamonds.csv


# Valuation of Feature Selection

In [179]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import numpy as np

In [180]:
# Split the original encoded data
X_train_all, X_test_all, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)
# Select features using f-regression selection
X_train_sel, X_test_sel = X_train_all[selected_features], X_test_all[selected_features]

# Model using all features
model_all = LinearRegression()
model_all.fit(X_train_all, y_train)
y_pred_all = model_all.predict(X_test_all)

# Model using selected features
model_sel = LinearRegression()
model_sel.fit(X_train_sel, y_train)
y_pred_sel = model_sel.predict(X_test_sel)

# Evaluate model performance
print("\nModel with all features:")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_all)))
print("R^2:", r2_score(y_test, y_pred_all))

print("\nModel with selected features:")
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_sel)))
print("R^2:", r2_score(y_test, y_pred_sel))



Model with all features:
RMSE: 1383.1259597573278
R^2: 0.8773928261441379

Model with selected features:
RMSE: 1510.6696243488689
R^2: 0.8537380328576377
